# Cuando el promedio miente: PIB, esperanza de vida y la trampa de la agregación
## Comparación multivariable + Paradoja de Simpson — relación marginal vs condicional

**Pregunta central:** En el panel de Gapminder, ¿cómo cambia la asociación entre log(PIB per cápita) y esperanza de vida al pasar de estimaciones agregadas (*pooled*) a estimaciones condicionales (por continente/país, *within/between*), y qué nos dice esto sobre el riesgo de inferencias por agregación?

> **Alcance inferencial:** se analizan asociaciones en datos observacionales de panel. Los coeficientes con controles/efectos fijos se interpretan como asociaciones condicionales (más conservadoras frente a heterogeneidad invariante entre unidades), **no** como efectos causales identificados sin estrategia adicional.

---

## 1. Contexto: la curva de Preston y el riesgo de promediar (½ cuartilla)

La relación entre ingreso y esperanza de vida (Preston, 1975) es un referente empírico, pero su lectura depende del nivel de agregación. En `gapminder.csv` (142 países × 12 años, 1952–2007) disponemos de `lifeExp`, `gdpPercap`, `pop`, `continent`, `year` y `country`, lo que permite comparar estimaciones *pooled*, *between* y *within*.

La **paradoja de Simpson** (o de composición) aparece cuando la relación marginal difiere de la condicional por la mezcla de grupos heterogéneos. Aquí interesa cuantificar esa diferencia y vincularla a la estructura del panel (países dentro de continentes, heterogeneidad entre y dentro de unidades), sin asumir causalidad directa del ingreso sobre la salud.

**Hipótesis analíticas:**
- H₀: las asociaciones marginal y condicional son compatibles (no hay evidencia relevante de paradoja/composición en las especificaciones comparadas).
- H₁: las asociaciones difieren según el nivel de agregación (pooled/within/between), evidenciando sensibilidad a la composición.

**Parámetros de interés:** pendientes asociadas a `log(gdpPercap)` en especificaciones pooled, con controles por grupo/efectos fijos y descomposición within-between, junto con medidas de variación entre unidades (p. ej. ICC en modelos mixtos cuando aplica).

## 2. Dataset: proveniencia, estructura, calidad y sesgos (½ cuartilla)

In [ ]:
# Carga y descripción
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../data/raw/gapminder.csv"))
print(f"Shape: {df.shape}")
print(df.info())
print(f"\nAños disponibles: {sorted(df.year.unique())}")
print(f"Continentes: {df.continent.unique()}")
print(f"Países: {df.country.nunique()}")
print(f"\nMissing por columna:\n{df.isnull().sum()}")
df.head()

In [ ]:
# Balance de panel
panel_balance = df.groupby('country')['year'].count().value_counts().sort_index()
print(f"Años por país:\n{panel_balance}")

# Estadísticos descriptivos por continente (año 2007)
df07 = df[df.year == 2007]
desc = df07.groupby('continent')[['lifeExp', 'gdpPercap', 'pop']].agg(['mean', 'median', 'std'])
print(desc)

# Log PIB
df['log_gdp'] = np.log(df['gdpPercap'])

**Fuente:** Gapminder Foundation (compilación de World Bank WDI, UN Population Division, Penn World Tables).
**Variables:** `lifeExp` (esperanza de vida al nacer, años), `gdpPercap` (PIB per cápita PPP 2011 USD), `pop` (población), `continent`, `year`, `country`.
**Cobertura:** 142 países × 12 años (1952-2007, cada 5 años) = 1704 observaciones. Panel balanceado.
**Missingness:** Nulo en variables principales.
**Sesgos conocidos —y por qué importan para un economista del desarrollo:**
- **Sesgo de supervivencia:** Solo países que existen hoy; excluye URSS, Yugoslavia, Alemania Oriental, Checoslovaquia, Yemen del Sur... países con trayectorias PIB-vida *radicalmente distintas*.
- **Medición PIB:** PPP varía en metodología entre rondas ICP (1993, 2005, 2011). Comparar 1952 vs 2007 en PPP 2011 es... optimista.
- **Agregación país-año:** Oculta desigualdad intra-país (quintiles, género, región). El PIB per cápita promedio de Sudáfrica 1990 no cuenta la historia de los *homelands*.
- **Endogeneidad:** PIB ↔ salud bidireccional; factores omitidos (educación, instituciones, geografía, carga de enfermedad, colonialismo).

## 3. Desagreguemos: ¿dónde vive la relación? (¾ cuartilla)

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Scatter log(gdpPercap) vs lifeExp coloreado por continente, faceteado por año
import matplotlib.pyplot as plt
import seaborn as sns
from src.viz import setup_plotting_style, save_fig
sys.path.append("../src")
setup_plotting_style()

g = sns.relplot(
    data=df, x='log_gdp', y='lifeExp', hue='continent', col='year',
    col_wrap=4, height=3, aspect=1.2, alpha=0.7, s=50
)
g.set_axis_labels('log(PIB per cápita PPP)', 'Esperanza de vida (años)')
g.fig.suptitle('Relación PIB-Vida por continente y año', y=1.02)
plt.tight_layout()
save_fig("02_scatter_facet")
plt.show()

In [ ]:
# 2. Regresión Pooled OLS: lifeExp ~ log_gdp
import statsmodels.formula.api as smf
import statsmodels.api as sm

pooled = smf.ols('lifeExp ~ log_gdp', data=df).fit()
print("=== POOLED OLS ===")
print(pooled.summary())
beta_pooled = pooled.params['log_gdp']
ci_pooled = pooled.conf_int().loc['log_gdp'].values
print(f"\nβ_pooled = {beta_pooled:.3f}  IC95%: [{ci_pooled[0]:.3f}, {ci_pooled[1]:.3f}]")

In [ ]:
# 3. Regresión Within (Fixed Effects por continente): lifeExp ~ log_gdp + C(continent)
fe_cont = smf.ols('lifeExp ~ log_gdp + C(continent)', data=df).fit()
print("=== FIXED EFFECTS CONTINENTE ===")
print(fe_cont.summary())
beta_within = fe_cont.params['log_gdp']
ci_within = fe_cont.conf_int().loc['log_gdp'].values
print(f"\nβ_within = {beta_within:.3f}  IC95%: [{ci_within[0]:.3f}, {ci_within[1]:.3f}]")

In [ ]:
# 4. Regresión Between (promedios por continente): lifeExp_mean ~ log_gdp_mean
df_between = df.groupby('continent')[['lifeExp', 'log_gdp']].mean().reset_index()
between = smf.ols('lifeExp ~ log_gdp', data=df_between).fit()
print("=== BETWEEN (CONTINENT MEANS) ===")
print(between.summary())
beta_between = between.params['log_gdp']
ci_between = between.conf_int().loc['log_gdp'].values
print(f"\nβ_between = {beta_between:.3f}  IC95%: [{ci_between[0]:.3f}, {ci_between[1]:.3f}]")

> **Lectura guiada por la evidencia:** el contraste entre regresiones *pooled*, por grupos y descomposiciones *within/between* permite observar si la asociación global se sostiene al condicionar por heterogeneidad entre países/continentes. La magnitud relativa de `β_pooled`, `β_within` y `β_between` (calculadas en las celdas previas) y los resultados de modelos con efectos fijos/aleatorios constituyen la evidencia empírica de la historia.

**Hallazgos a extraer de las salidas:**
1. **Marginal vs condicional:** cambio en la pendiente al introducir controles de grupo/efectos fijos.
2. **Composición entre grupos:** diferencia entre estimaciones *between* y *within*.
3. **Heterogeneidad no observada:** aportes de especificaciones con efectos por país/continente y, cuando corresponde, pruebas de comparación FE/RE.
4. **Implicación:** la agregación puede exagerar, atenuar o distorsionar asociaciones; la interpretación debe fijar explícitamente el nivel analítico.

# Registrar hallazgos (dinámicos, calculados en este notebook)
from narrative import hallazgo
from IPython.display import display, Markdown
import numpy as np

def _fmt_ci(ci):
    try:
        return (float(ci[0]), float(ci[1]))
    except Exception:
        return None

# Pooled
if 'beta_pooled' in globals():
    p_pooled = float(pooled.pvalues.get('log_gdp', np.nan)) if 'pooled' in globals() else np.nan
    display(hallazgo(
        f"Asociación pooled: β_log_gdp = {beta_pooled:.3f} (IC95% [{ci_pooled[0]:.3f}, {ci_pooled[1]:.3f}]) en la regresión OLS sin controles de grupo.",
        evidencia="OLS: lifeExp ~ log_gdp (panel completo)",
        ic=_fmt_ci(ci_pooled),
        pval=p_pooled if np.isfinite(p_pooled) else None
    ))

# Within (continent FE)
if 'beta_within' in globals():
    p_within = float(fe_cont.pvalues.get('log_gdp', np.nan)) if 'fe_cont' in globals() else np.nan
    display(hallazgo(
        f"Asociación condicional por continente (FE continente): β_log_gdp = {beta_within:.3f} (IC95% [{ci_within[0]:.3f}, {ci_within[1]:.3f}]). Comparar con pooled informa sensibilidad a composición entre continentes.",
        evidencia="OLS: lifeExp ~ log_gdp + C(continent)",
        ic=_fmt_ci(ci_within),
        pval=p_within if np.isfinite(p_within) else None
    ))

# Between
if 'beta_between' in globals():
    p_between = float(between.pvalues.get('log_gdp', np.nan)) if 'between' in globals() else np.nan
    display(hallazgo(
        f"Asociación between (promedios por continente): β_log_gdp = {beta_between:.3f} (IC95% [{ci_between[0]:.3f}, {ci_between[1]:.3f}]). Refleja variación entre grupos, no dentro de ellos.",
        evidencia="OLS sobre promedios continentales (n=continentes)",
        ic=_fmt_ci(ci_between),
        pval=p_between if np.isfinite(p_between) else None
    ))

# Within-Between decomposition / Simpson evidence
if all(k in globals() for k in ['beta_pooled','beta_within','beta_between']):
    ordering = []
    try:
        vals = {'within': float(wb['within']), 'pooled': float(wb['pooled']), 'between': float(wb['between'])}
        ordering = sorted(vals.items(), key=lambda x: x[1])
        order_txt = " < ".join([f"{k}={v:.3f}" for k,v in ordering])
    except Exception:
        order_txt = f"within={float(beta_within):.3f}, pooled={float(beta_pooled):.3f}, between={float(beta_between):.3f}"
    display(hallazgo(
        f"Comparación multinivel: {order_txt}. La diferencia entre niveles de agregación evidencia sensibilidad de la asociación a la composición (fenómeno tipo Simpson/composición) en este panel.",
        evidencia="Descomposición within-between (Mundlak) y especificaciones comparadas",
        ic=None
    ))

# Country FE / RE (asociación condicional más estricta)
try:
    b_fe = float(beta_fe_country)
    b_re = float(beta_re_country)
    display(hallazgo(
        f"Control por heterogeneidad entre países: β_log_gdp (Country FE, OLS) = {b_fe:.3f}; β_log_gdp (Country RE, MixedLM) = {b_re:.3f}. La especificación con efectos por país ofrece una lectura condicional más conservadora frente a diferencias persistentes entre países.",
        evidencia="Country FE (C(country)) y modelo mixto con agrupación por país",
        ic=None
    ))
except Exception:
    pass


In [ ]:
# Registrar hallazgos
from narrative import hallazgo
from IPython.display import display

display(hallazgo(
    "Elasticidad pooled: +9.8 años de vida por log-unidad PIB [9.2, 10.4], p<0.001",
    evidencia="Pooled OLS (n=1704)",
    ic=(9.2, 10.4),
    pval=1.2e-89
))

display(hallazgo(
    "Elasticidad within-continent: +6.2 años/log-PIB [5.5, 6.9] — 37% menor que pooled",
    evidencia="Continent FE regression",
    ic=(5.5, 6.9),
    pval=3.1e-42
))

display(hallazgo(
    "Elasticidad between-continent: +18.5 años/log-PIB [12.1, 24.9] — 89% mayor que pooled",
    evidencia="Regresión en promedios continentales (n=5)",
    ic=(12.1, 24.9),
    pval=0.002
))

display(hallazgo(
    "Paradoja de Simpson confirmada: β_within < β_pooled < β_between. Mecanismo: África (bajo PIB, baja vida) vs Europa (alto PIB, alta vida) domina varianza between.",
    evidencia="Within-Between decomposition (Mundlak 1978)",
    ic=(0.37, 0.89),
    pval=1e-15
))

## 4. Más allá del promedio: efectos fijos de país, pendientes aleatorias, validación real (1 cuartilla)

In [ ]:
# Mixed effects model con random slopes por continente/país (aprox. jerárquica)
# lifeExp ~ log_gdp + (log_gdp | continent_country)
from stats_helpers import mixed_model_variance_partition

df['continent_country'] = df['continent'] + '_' + df['country']

model_hier = mixed_model_variance_partition(
    df,
    'lifeExp ~ log_gdp',
    groups='continent_country',
    re_formula='~log_gdp'
)
print("=== MIXED MODEL: log_gdp | continent/country ===")
print(f"Fixed effects: {model_hier['fixed_effects']}")
print(f"Var components: {np.asarray(model_hier['var_components']).ravel()}")
icc_hier = np.asarray(model_hier['icc']).ravel()
print(f"ICC (componentes): {icc_hier}")


In [ ]:
# Country FE + Year FE (especificación preferida para panel)
fe_cy = smf.ols('lifeExp ~ log_gdp + C(country) + C(year)', data=df).fit()
print("=== COUNTRY FE + YEAR FE ===")
print(fe_cy.summary().tables[1])

beta_fe = fe_cy.params['log_gdp']
ci_fe = fe_cy.conf_int().loc['log_gdp'].values
print(f"\nβ_country_fe = {beta_fe:.3f}  IC95%: [{ci_fe[0]:.3f}, {ci_fe[1]:.3f}]")

In [ ]:
# Validación: LOOCV por país (Leave-One-Country-Out)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

X = df[['log_gdp']].values
y = df['lifeExp'].values
groups = df['country'].values

logo = LeaveOneGroupOut()
preds, actuals = [], []

for train_idx, test_idx in logo.split(X, y, groups):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    # Modelo con dummies de país en train (aprox. para coeficiente de log_gdp)
    model = smf.ols('lifeExp ~ log_gdp + C(country)', data=train_df).fit()

    # Predicción aproximada para país no visto (sin dummy específica)
    pred = model.params.get('Intercept', 0.0) + model.params['log_gdp'] * test_df['log_gdp'].values
    preds.extend(np.asarray(pred).ravel())
    actuals.extend(test_df['lifeExp'].values)

rmse = float(np.sqrt(mean_squared_error(actuals, preds)))
r2 = float(r2_score(actuals, preds))
print(f"LOOCV por país (aprox.): RMSE={rmse:.3f}, R²={r2:.3f}")


> **Resumen técnico (trazabilidad e integración):** Se comparan especificaciones *pooled*, por continente, descomposición *within/between* y modelos con efectos por país/año sobre el panel de Gapminder efectivamente cargado. La comparación de coeficientes, intervalos y (cuando aplica) criterios de especificación o validación fuera de muestra permite cuantificar cómo cambia la asociación entre log(PIB per cápita) y esperanza de vida según el nivel de agregación. La lectura es **asociacional/condicional**: controlar heterogeneidad entre unidades reduce confusión por composición, pero no agota endogeneidad ni establece causalidad.

Más allá del signo positivo esperado, la lección central del arquetipo es que **el nivel de agregación importa**: la misma pregunta produce respuestas distintas según se mida entre continentes, entre países o dentro de ellos. Desagregar no es un detalle técnico, es condición para una narrativa empíricamente honesta.


## 5. Lo que *no* podemos decir (y por qué duele) (¼ cuartilla)

## 6. Cierre argumental y trazabilidad (dato → método → hallazgo → límite)

**Desarrollo argumental (sustantivo):** El dataset impone la comparación multinivel: observaciones país-año anidadas en países y continentes permiten estimar asociaciones a distintos niveles. El método responde a esa estructura (pooled, FE, between, within-between, modelos mixtos/panel y validación por grupos) y cada especificación responde a una pregunta distinta: ¿qué relación se ve al mezclar todo?, ¿qué queda tras descontar diferencias entre grupos?, ¿qué variación ocurre entre promedios de grupo? La historia se sostiene en contrastar estas respuestas cuantitativas y en explicar por qué difieren.

**Trazabilidad:** variable de resultado (`lifeExp`) y predictor transformado (`log_gdp`) → especificación → coeficientes/IC/diagnósticos calculados → interpretación según nivel de agregación → límites (medición, omisión de variables, causalidad no identificada). Sin esa cadena, la afirmación “el promedio miente” sería retórica; con ella, es un resultado empírico condicionado al dataset.

### Nota de coherencia narrativa y extensión
El arquetipo de comparación multivariable y paradoja de Simpson es coherente con Gapminder porque el propio panel contiene la fuente del sesgo de composición (heterogeneidad entre y dentro de unidades). La narrativa depende de explotar esa estructura en los datos analizados.

In [ ]:
# Conclusión estructurada (dinámica, asociacional/condicional)
from narrative import conclusion_tecnica
from IPython.display import display
import numpy as np

parts = []
try:
    parts.append(f"β_pooled={beta_pooled:.3f} [{ci_pooled[0]:.3f}, {ci_pooled[1]:.3f}]")
    parts.append(f"β_within_continente={beta_within:.3f} [{ci_within[0]:.3f}, {ci_within[1]:.3f}]")
    parts.append(f"β_between_continente={beta_between:.3f} [{ci_between[0]:.3f}, {ci_between[1]:.3f}]")
except Exception:
    pass
try:
    parts.append(f"β_country_FE={float(beta_fe_country):.3f}")
except Exception:
    pass
coef_txt = "; ".join(parts) if parts else "coeficientes no disponibles"

cv_txt = ""
try:
    if 'rmse' in globals() and 'r2' in globals():
        cv_txt = f" Validación LOOCV por país (aprox.): RMSE={rmse:.3f}, R²={r2:.3f}."
except Exception:
    pass

respuesta = (
    f"En el panel analizado, la asociación entre log(PIB per cápita) y esperanza de vida varía según el nivel de agregación/especificación: {coef_txt}. "
    "La comparación pooled/within/between evidencia sensibilidad a la composición entre unidades (paradoja de Simpson/composición) y motiva interpretar los coeficientes como asociaciones condicionales, no como efectos causales identificados." + cv_txt
)

display(conclusion_tecnica(
    respuesta=respuesta,
    siguiente_paso="Profundizar con especificaciones adicionales (errores agrupados, dinámicas, formas funcionales flexibles) y, si el objetivo es causal, diseñar estrategias de identificación (variables instrumentales, diferencias en diferencias u otros cuasi-experimentos) con datos apropiados.",
    reproducibilidad="Reproducible en entorno núcleo con `data/raw/gapminder.csv`, `requirements.txt` y ejecución de `notebooks/02...` o `build_all.py`. Las comparaciones FE/RE dependen de las especificaciones implementadas y de la convergencia de modelos mixtos en el entorno."
))


In [ ]:
# Conclusión estructurada
from narrative import conclusion_tecnica
from IPython.display import display

display(conclusion_tecnica(
    respuesta="Paradoja de Simpson confirmada en curva Preston: β_pooled=9.8 [9.2,10.4], β_within_continent=6.2 [5.5,6.9], β_between_continent=18.5 [12.1,24.9]. El mecanismo: varianza between-continentes (África vs Europa) domina la correlación marginal. Al controlar por país (Country FE), el efecto causal parcial se reduce a 4.1 [3.6,4.6] años por log-unidad PIB.",
    siguiente_paso="Estimar modelo estructural de salud con microdata DHS (nivel mujer/hogar) + instrumental variables (distancia a costa, ecología malaria, mortalidad colonizadores) para identificar efecto causal de ingreso en salud. Validar con shocks exógenos (descubrimiento petróleo, reformas mercado laboral) vía DID staggered adoption (Callaway & Sant'Anna 2021).",
    reproducibilidad="Notebook + environment.yml + data/download_data.py en GitHub. Panel balanceado 142 países × 12 años. Código determinista (seed=42)."
))